# Day 2 — Hands-On Lab 1: Storage Credentials, External Locations & Lakeflow Connect

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 2 — Lakeflow Connect + Storage Credentials & External Locations |
| **Source** | Your own Supabase Postgres project (`orders`, `order_items`) + your own ADLS storage account |
| **Duration** | 60 minutes |
| **Output** | A working Storage Credential + External Location, and a live Lakeflow Connect query-based CDC pipeline |

### Learning Objectives
- Create your own Storage Credential backed by an Azure Managed Identity
- Create your own External Location and verify access with zero hardcoded keys
- Create your own PostgreSQL connection to your own Supabase project
- Configure a Lakeflow Connect ingestion pipeline (query-based, cursor column) for `orders` and `order_items`
- Prove CDC: change a row in Supabase, re-run the pipeline, confirm only the change syncs
- Prove the trade-off: hard-delete a row and confirm query-based mode does **not** detect it

---
**Instructions:** Run each cell with **Shift + Enter**. Phases A–B are mostly Azure Portal / Databricks UI steps — read carefully, screenshot as you go for your submission. Phases C–D are notebook cells.

---
## Phase A — Create Your Own Storage Credential

**Goal:** Register a Storage Credential in Unity Catalog backed by an Azure Managed Identity — same pattern as the real `ecomprojectscredentials` shown in ILT 2, but naming everything after yourself.

### A1 — Find (or create) your Access Connector

```
Azure Portal → search bar → "Access Connectors for Azure Databricks"
  → find the Access Connector linked to your Databricks workspace
  → Settings → Properties → copy the Resource ID
    (looks like: /subscriptions/.../resourceGroups/.../providers/Microsoft.Databricks/accessConnectors/...)

If none exists:
  Azure Portal → Create a resource → search "Azure Databricks Access Connector"
  → create it in the same Resource Group as your Databricks workspace
```

### A2 — Confirm the Access Connector has RBAC on your storage account

```
Azure Portal → your storage account (from Day 1, e.g. globalmart<yourname>)
  → Access Control (IAM) → Role assignments
  → confirm the Access Connector has: Storage Blob Data Contributor

If missing:
  → Add role assignment → Storage Blob Data Contributor
  → Assign access to: Managed Identity → select your Access Connector
```

### A3 — Create the Storage Credential in Databricks

```
Databricks → Catalog icon (left sidebar) → External Data → Storage Credentials
  → Create credential

  Credential type   : Azure Managed Identity
  Credential name   : <yourname>_credential        e.g. virinchy_credential
  Access Connector ID: (paste the Resource ID from A1)
  Comment           : Managed identity for my Day 1 storage account
  → Create
```

**Verify:** the credential appears in the list with status **Active**.

---
## Phase B — Create Your Own External Location

**Goal:** Map your ADLS container to the Storage Credential from Phase A — mirrors the real `gbmart-ext-loc` shown in ILT 2.

### B1 — Create the External Location

```
Databricks → Catalog icon → External Data → External Locations
  → Create location

  External location name : <yourname>_external        e.g. virinchy_external
  URL                     : abfss://<your-container>@<your-storage-account>.dfs.core.windows.net/
  Storage credential      : <yourname>_credential      (from Phase A)
  Comment                 : External location for my Day 1 storage account
  → Create
```

### B2 — Test the connection

```
Click "Test connection" on the External Location you just created.
Expected: ✅ Connection test succeeded — lists files under your container.

If you see a permissions error → go back to A2 and re-check the RBAC role assignment.
```

Fill in your own values below, then run the cells to verify access **with zero hardcoded keys**.

In [ ]:
# ─── B3: Verify access — NO spark.conf.set, NO storage key ───────────────────
# Replace with your own container / storage account from Day 1

your_container       = "amazon-data"                      # ← your container name
your_storage_account = "YOUR_STORAGE_ACCOUNT_NAME"         # ← your storage account from Day 1

base_path = f"abfss://{your_container}@{your_storage_account}.dfs.core.windows.net"

print("Listing raw/ — no storage key needed, Unity Catalog handles auth:")
files = dbutils.fs.ls(f"{base_path}/raw/")
for f in files:
    print(f"  {f.name:<40} {f.size/1024:>8.1f} KB")

print(f"\nTotal files: {len(files)}")

In [ ]:
# ─── B4: Read a real file through the External Location ──────────────────────

customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{base_path}/raw/customers_010626.csv")
)

print(f"customers_010626.csv — {customers_df.count():,} rows, {len(customers_df.columns)} columns")
customers_df.show(3, truncate=False)

In [ ]:
# ─── B5: Verify via SQL — confirm your External Location is registered ───────

your_external_location = "YOUR_NAME_external"   # ← e.g. virinchy_external

spark.sql("SHOW EXTERNAL LOCATIONS").show(truncate=False)
spark.sql(f"DESCRIBE EXTERNAL LOCATION {your_external_location}").show(truncate=False)

---
## Phase C — Create Your Own Lakeflow Connect Pipeline

**Goal:** Mirror the real `ecom_gbmart_conn` → `orders_data_ingestion_cdc` pipeline from ILT 2 with your own — pointing at your own Supabase project — then stand up a CDC pipeline for `orders` and `order_items`.

> **Prerequisite:** you should already have your own Supabase project with `orders` and `order_items` tables populated (same project pattern used in earlier Supabase hands-ons — each student runs their own). **Check that both tables have an `updated_at` column that actually gets refreshed on UPDATE** — the real pipeline uses a query-based connector keyed on `updated_at`, so without it, incremental sync has nothing to detect changes with. If it's missing, add it now:
> ```sql
> ALTER TABLE orders ADD COLUMN IF NOT EXISTS updated_at TIMESTAMP DEFAULT NOW();
> ALTER TABLE order_items ADD COLUMN IF NOT EXISTS updated_at TIMESTAMP DEFAULT NOW();
> ```

### C1 — Create the Connection

```
Databricks → Catalog icon → Connections → + Add connection

  Connection name : postgresql_<yourname>          e.g. postgresql_virinchy
  Type            : PostgreSQL
  Host            : (from your Supabase project → Settings → Database)
                     pattern: aws-0-<region>.pooler.supabase.com
  Port            : 5432
  Database        : postgres
  Username        : postgres.<your_project_ref>
  Password        : (from your Supabase credentials)
  SSL             : Require
  → Test connection → Create
```

### C2 — Create the Ingestion Pipeline

```
Databricks → Data Ingestion → PostgreSQL (Preview)

  Connection : postgresql_<yourname>
  Database   : postgres
  Schema     : public
  Tables     : ✅ orders
               ✅ order_items

  Destination (Unity Catalog):
    Catalog : <your_catalog>            ← your own Unity Catalog catalog
    Schema  : bronze
    Tables  : auto-created — one per source table

  Table configuration (per table):
    Primary key    : order_id / order_item_id (or whatever your own schema calls them)
    Connector mode : Query-based
    Cursor column  : updated_at         ← same mode as the real GlobalMart pipeline

  → Create Pipeline → Start
```

### C3 — Record the first-run result

```
Status    : Completed?  ___________
orders      Upserted:   ___________
order_items Upserted:   ___________
Duration    :           ___________
```

In [ ]:
# ─── C4: Verify the pipeline landed in Unity Catalog ──────────────────────────
# Replace with your own catalog name from Phase C2

your_catalog = "YOUR_CATALOG_NAME"   # ← the catalog you picked as pipeline destination

orders_count = spark.sql(f"SELECT COUNT(*) AS n FROM {your_catalog}.bronze.orders").collect()[0]["n"]
items_count  = spark.sql(f"SELECT COUNT(*) AS n FROM {your_catalog}.bronze.order_items").collect()[0]["n"]

print(f"orders      : {orders_count:,} rows")
print(f"order_items : {items_count:,} rows")

spark.sql(f"DESCRIBE EXTENDED {your_catalog}.bronze.orders").show(truncate=False)
# Look for: Type = STREAMING_TABLE, Catalog Name = your_catalog (not hive_metastore)

---
## Phase D — Prove CDC (and Its Limit)

**Goal:** Make a change in Supabase, re-run the pipeline, and show that only the change syncs — not the whole table. Then prove the query-based trade-off from ILT 2 for yourself: a hard DELETE does **not** sync.

> We only test INSERT and UPDATE below, on purpose — your pipeline is query-based on `updated_at`, the same mode as GlobalMart's real `orders_data_ingestion_cdc`. A DELETE has no `updated_at` value to trigger on, so it can't be detected this way. Phase D's bonus step (D5) proves this directly instead of just asserting it.

### D1 — Make a change in Supabase SQL Editor

```sql
-- Insert 1 new order
INSERT INTO orders (order_id, customer_id, order_date, status, payment_method_id, shipping_tier_id)
VALUES ('O-TEST-001', 'CUST-99999', NOW(), 'pending', 'PM-001', 'STD');

-- Update it — simulate it shipping
UPDATE orders SET status = 'shipped' WHERE order_id = 'O-TEST-001';
```

### D2 — Re-run the pipeline

```
Databricks → Jobs & Pipelines → postgresql_<yourname> → Start

Expected result:
  Status   : Completed
  Upserted : 1        ← only the 1 changed row, not the whole orders table
  Duration : faster than the first run
```

### D3 — Confirm the row in Unity Catalog

In [ ]:
# ─── D4: Find the test row — prove it landed via CDC, not a full reload ───────

result_df = spark.sql(f"""
    SELECT order_id, customer_id, status
    FROM {your_catalog}.bronze.orders
    WHERE order_id = 'O-TEST-001'
""")
result_df.show(truncate=False)

new_orders_count = spark.sql(f"SELECT COUNT(*) AS n FROM {your_catalog}.bronze.orders").collect()[0]["n"]
print(f"orders row count now: {new_orders_count:,}  (was {orders_count:,} before D1 — should be +1)")
print("status should read 'shipped' — the UPDATE was captured, not just the INSERT")

---
### D5 (Bonus) — Prove the DELETE Blind Spot

Now delete the test row entirely in Supabase and watch what (doesn't) happen.

```sql
-- In Supabase SQL Editor:
DELETE FROM orders WHERE order_id = 'O-TEST-001';
```

Re-run your pipeline (same as D2), then run the verification cell below.

In [ ]:
# ─── D6: Verify the DELETE did NOT propagate — the query-based blind spot ─────
# This is expected behavior, not a bug: a query-based connector on `updated_at`
# has no way to notice a row disappeared, since a DELETE leaves no row behind
# to carry an updated_at value. Only log-based (WAL) replication would catch this.

still_there_df = spark.sql(f"""
    SELECT order_id, customer_id, status
    FROM {your_catalog}.bronze.orders
    WHERE order_id = 'O-TEST-001'
""")

row_count = still_there_df.count()
if row_count > 0:
    print("Confirmed: the deleted row is STILL in Bronze — the pipeline has no way to know it's gone.")
    still_there_df.show(truncate=False)
else:
    print("Row not found — if your source table doesn't actually hard-delete "
          "(e.g. a trigger soft-deletes instead), this result won't show the blind spot.")

---
## Submission Checklist

> ⚠️ **Replace any real password/secret values with placeholders before uploading this notebook.**

```
Submission Checklist
────────────────────────────────────────────────────────────────
✅ Access Connector confirmed / created, Resource ID copied
✅ Access Connector has Storage Blob Data Contributor on my storage account
✅ Storage Credential created: <yourname>_credential (status: Active)
✅ External Location created: <yourname>_external
✅ "Test connection" succeeded on the External Location
✅ dbutils.fs.ls() listed raw/ files — no storage key used
✅ customers CSV read successfully through the External Location
✅ SHOW EXTERNAL LOCATIONS / DESCRIBE EXTERNAL LOCATION confirmed
✅ Own Supabase project has orders + order_items tables populated
✅ Confirmed both tables have a live-updating updated_at column
✅ PostgreSQL Connection created: postgresql_<yourname>
✅ Ingestion Pipeline created (Query-based, cursor: updated_at) and first run completed
── orders row count (first run):        ______
── order_items row count (first run):   ______
✅ CDC proof: inserted + updated 1 row in Supabase
✅ Re-ran pipeline — confirmed Upserted: 1 (not a full reload)
── orders row count (after CDC run):     ______
✅ Bonus: deleted the test row in Supabase, re-ran pipeline, confirmed it's STILL in Bronze
✅ Screenshot: Storage Credential + External Location pages
✅ Screenshot: Ingestion Pipeline run history (all runs, including the DELETE test)
✅ Notebook uploaded with all secrets replaced by placeholders
────────────────────────────────────────────────────────────────
```